# Praktikum Machine Learning: Klasifikasi Pay Band

**Nama:** Hilal Tsabitul Azmi Arba'i

**NRP:** 5054251008

Template Kaggle Competition Praktikum Machine Learning (materi Klasifikasi).


## Setup

In [ ]:
# Import Library
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, ConfusionMatrixDisplay
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression


## Load Data

In [ ]:
# Sesuaikan path dengan lokasi file kalian (Kaggle: /kaggle/input/<nama-kompetisi>/...)
df_train = pd.read_csv('train (4).csv')
df_test = pd.read_csv('test (2).csv')

print("Train Data Shape:", df_train.shape)
print("Test Data Shape:", df_test.shape)

df_train.head(50)

Train Data Shape: (60430, 21)
Test Data Shape: (29070, 20)


,employee_id,source_dataset,import_batch,age,gender,ethnicity,education_level,years_experience,tenure_years,department,...,country,remote_status,performance_rating,performance_review_score,hire_date,currency,salary_raw,record_hash,notes,pay_band
0,EMP-194754,hris_export_2023,24,NaN,?,NaN,bachelor's degree,NaN,6.0,Engineering,...,PH,Hybrid,exceeds,3.1,04-30-2020,PHP,"191,210 per month",fedab652e9c8,NaN,Band 2
1,EMP-109967,hris_export_2023,22,0.0,female,White,Associate Degree,24.0,0.0,IT,...,Brazil,On site,Meets Expectations,4.1,02 Apr 2026,BRL,"BRL 250,722",a94720bee9e0,NaN,Band 2
2,EMP-128419,hris_export_2023,8,33.0,FEMALE,unknown,PhD,12.0,1.0,Legal,...,United Kingdom,Remote,Meets Expectations,2.1,23 Dec 2024,?,"28,276.28",8f8b484b87ee,NaN,Band 1
3,EMP-108690,hris_export_2023,39,NaN,Woman,Asian,PhD,20.0,4.0,Legal,...,Brazil,On-site,Meets Expectations,4.1,2022-07-20,BRL,18315 monthly,a7a6d7f3cc91,NaN,Band 1
4,EMP-169742,acquired_company_legacy,22,39.0,Other,White,bachelor's degree,-1.0,4.0,Data Science Department,...,USA,Hybrid,NaN,3.5,"March 31, 2022",USD,"20.855,27 USD",7826c5135f34,NaN,Band 1
5,EMP-189049,acquired_company_legacy,13,NaN,Woman,Black or African American,MSc,-1.0,2.0,Legal Department,...,France,Remote,NaN,4.4,2024-07-17,EUR,20620,75ca4e7688bc,TBD,Band 1
6,EMP-125167,survey_response_2024,14,NaN,NaN,White,BSc,24.0,4.0,Sales,...,France,Onsite,Exceeds Expectations,3.4,18 Feb 2022,EUR,"48,455.37",1a5c9dcb6761,NaN,Band 3
7,EMP-128093,acquired_company_legacy,19,54.0,M,Black or African American,BSc,31.0,5.0,Operations,...,Kenya,Onsite,Meets Expectations,4.8,02-20-2021,KES,"8,924,798 KES",f4728e19d94c,NaN,Band 3
8,EMP-109731,acquired_company_legacy,20,54.0,F,Black or African American,bachelor's degree,NaN,8.0,Operations,...,US,Onsite,exceeds,2.8,2017-07-01,USD,"4,177 per month",50c8a6e3b611,NaN,Band 3
9,EMP-196010,hris_export_2023,7,20.0,F,White,PhD.,NaN,0.0,HR,...,Brazil,On-site,exceeds,2.8,08 Aug 2026,BRL,"18,969 per month",2d5bcb1e0e1f,NaN,Band 2


## Exploratory Data Analysis

Distribusi target tidak seimbang (Band 4+ ~2%), banyak outlier numerik, dan `salary_raw` berformat campuran — dicek di bawah.


In [ ]:
# EDA ringkas: distribusi target, statistik numerik, missing, outlier
print("Distribusi pay_band:")
print(df_train["pay_band"].value_counts(normalize=True).round(4))
print(df_train["pay_band"].value_counts())
display(df_train.describe(include="all").T)
display(df_train.isnull().sum().sort_values(ascending=False).to_frame("missing"))
# Contoh outlier dari describe sebelumnya: age 0/-1/999, tenure negatif, review_score >5
for c in ["age","years_experience","tenure_years","performance_review_score"]:
    s = pd.to_numeric(df_train[c], errors="coerce")
    print(f"{c}: min={s.min()}, p50={s.median()}, max={s.max()}, n_missing={s.isna().sum()}")
print("Contoh salary_raw per band:")
print(df_train.groupby("pay_band")["salary_raw"].apply(lambda s: s.sample(min(3, len(s)), random_state=0).tolist()))


## Cleaning + Feature Engineering

Perbaikan: (1) JANGAN buang `salary_raw/age/years_experience/performance_review_score` — dibersihkan; (2) buang kolom bocor/ID `employee_id, record_hash, notes, source_dataset, import_batch`; (3) fungsi `clean_df` dipakai ke train DAN test agar konsisten; (4) placeholder `?, -, unknown` -> NaN; (5) normalisasi kategori berantakan (country, gender, dept, job_title, dsb).


In [ ]:
import re
PLACEHOLDERS = {"?", "-", "unknown", "Unknown", "UNKNOWN", "", " ", "nan", "NaN", "none", "None", "n/a", "N/A", "NA", "TBD"}

def clean_placeholders(s):
    if pd.isna(s):
        return np.nan
    t = str(s).strip()
    return np.nan if t in PLACEHOLDERS else t

def parse_salary(s):
    """Return (nilai_gaji, is_monthly). Handle '46.4k', '20.855,27 USD' (desimal Eropa), '191,210 per month'."""
    if pd.isna(s):
        return np.nan, 0
    low = str(s).lower()
    is_monthly = 1 if ("month" in low or "/month" in low or "monthly" in low) else 0
    m_k = re.search(r"([\d.,]+\s*k)", low)
    if m_k:  # format kilo, cth '46.4k', '1,466.4k'
        try:
            return float(m_k.group(1).replace("k", "").replace(",", "").strip()) * 1000, is_monthly
        except ValueError:
            return np.nan, is_monthly
    tmp = re.sub(r"(per month|/month|monthly|per annum|per year)", " ", low)
    for cur in ["brl","usd","eur","gbp","kes","cad","zar","ngn","inr","sgd","aud","php"]:
        tmp = tmp.replace(cur, " ")
    tmp = tmp.replace("£", " ").replace("$", " ").replace("€", " ")
    m = re.search(r"[\d.,]+", tmp)
    if not m:
        return np.nan, is_monthly
    num = m.group(0)
    try:
        if "." in num and "," in num:
            num = num.replace(".", "").replace(",", ".") if num.rfind(",") > num.rfind(".") else num.replace(",", "")
        elif "," in num:
            parts = num.split(",")
            num = (parts[0] + "." + parts[1]) if (len(parts) == 2 and len(parts[1]) <= 2) else num.replace(",", "")
        return float(num), is_monthly
    except ValueError:
        return np.nan, is_monthly

COUNTRY_MAP = {"usa":"USA","us":"USA","united states":"USA","uk":"UK","united kingdom":"UK",
 "brasil":"Brazil","br":"Brazil","brazil":"Brazil","ph":"Philippines","phillipines":"Philippines",
 "philippines":"Philippines","ca":"Canada","canada":"Canada","au":"Australia","aus":"Australia",
 "australia":"Australia","za":"South Africa","s. africa":"South Africa","south africa":"South Africa",
 "ng":"Nigeria","nigeria":"Nigeria","ke":"Kenya","kenya":"Kenya","nl":"Netherlands","holland":"Netherlands",
 "netherlands":"Netherlands","in":"India","bharat":"India","india":"India","de":"Germany",
 "deutschland":"Germany","germany":"Germany","sg":"Singapore","singapore":"Singapore","fr":"France","france":"France"}

def clean_country(s):
    if pd.isna(s): return np.nan
    t = str(s).strip()
    if t in PLACEHOLDERS: return np.nan
    return COUNTRY_MAP.get(t.lower(), t.title())

def clean_gender(s):
    if pd.isna(s): return np.nan
    t = str(s).strip().lower()
    if t in {"?", "-", "unknown", ""}: return np.nan
    if t in ("m","male","man","mal"): return "M"
    if t in ("f","female","woman","femal"): return "F"
    if t in ("nb","non-binary","nonbinary","non binary"): return "NB"
    if t == "other": return "Other"
    return np.nan  # 'prefer not to say' dsb -> NaN

def clean_remote(s):
    if pd.isna(s): return np.nan
    t = str(s).strip().lower()
    if t in PLACEHOLDERS: return np.nan
    if "hybrid" in t: return "Hybrid"
    if "remote" in t: return "Remote"
    if "site" in t or t in ("onsite", "on-site", "on site"): return "Onsite"
    return np.nan

def clean_perf_rating(s):
    if pd.isna(s): return np.nan
    t = str(s).strip().lower()
    if t in PLACEHOLDERS: return np.nan
    if "below" in t: return "Below"
    if "exceed" in t: return "Exceeds"
    if "meet" in t: return "Meets"
    return np.nan

def clean_edu(s):
    if pd.isna(s): return np.nan
    t = str(s).strip()
    if t in PLACEHOLDERS: return np.nan
    l = t.lower().replace(".", "").strip()
    if "phd" in l: return "PhD"
    if "master" in l or l == "msc": return "Master"
    if "associate" in l: return "Associate"
    if "high school" in l: return "High School"
    if "bachelor" in l or l in ("bsc","bs"): return "Bachelor"
    return np.nan

def clean_dept(s):
    if pd.isna(s): return np.nan
    t = re.sub(r"\s+", " ", str(s).strip().lower().replace(".", ""))
    return re.sub(r"\s+(department|dept|team)$", "", t).strip()

def clean_job(s):
    if pd.isna(s): return np.nan
    return re.sub(r"\s+", " ", str(s).strip().lower().replace(".", "")).strip()

def clean_df(df):
    df = df.copy()
    for c in df.columns:  # placeholder -> NaN dulu agar imputer tidak menganggap '?' kategori
        if df[c].dtype == object:
            df[c] = df[c].apply(clean_placeholders)
    df["age"] = pd.to_numeric(df["age"], errors="coerce")
    df.loc[(df["age"] < 18) | (df["age"] > 70), "age"] = np.nan
    df["years_experience"] = pd.to_numeric(df["years_experience"], errors="coerce")
    df.loc[(df["years_experience"] < 0) | (df["years_experience"] > 50), "years_experience"] = np.nan
    df["tenure_years"] = pd.to_numeric(df["tenure_years"], errors="coerce")
    df.loc[(df["tenure_years"] < 0) | (df["tenure_years"] > 20), "tenure_years"] = np.nan
    df["performance_review_score"] = pd.to_numeric(df["performance_review_score"], errors="coerce")
    df.loc[(df["performance_review_score"] < 1) | (df["performance_review_score"] > 5), "performance_review_score"] = np.nan
    parsed = df["salary_raw"].apply(parse_salary)  # FITUR PENTING: jangan di-drop
    df["salary_value"] = parsed.apply(lambda x: x[0])
    df["salary_is_monthly"] = parsed.apply(lambda x: x[1])
    # FIX: annualize agar bulanan vs tahunan sebanding (median monthly ~5k vs annual ~62k)
    df["salary_annual"] = df["salary_value"] * np.where(df["salary_is_monthly"] == 1, 12.0, 1.0)
    df["salary_log"] = np.log1p(df["salary_annual"].clip(lower=0))
    df["gender"] = df["gender"].apply(clean_gender)
    df["country"] = df["country"].apply(clean_country)
    df["remote_status"] = df["remote_status"].apply(clean_remote)
    df["performance_rating"] = df["performance_rating"].apply(clean_perf_rating)
    df["education_level"] = df["education_level"].apply(clean_edu)
    df["department"] = df["department"].apply(clean_dept)
    df["job_title"] = df["job_title"].apply(clean_job)
    df["currency"] = df["currency"].str.strip().str.upper().replace(list(PLACEHOLDERS), np.nan)
    df["ethnicity"] = df["ethnicity"].apply(lambda s: np.nan if pd.isna(s) else str(s).strip().title())
    # FIX: format campuran ("04-30-2020","02 Apr 2026","20220226") butuh format="mixed" (default hanya ~20% terparse)
    hd = pd.to_datetime(df["hire_date"], errors="coerce", format="mixed")
    df["hire_year"] = hd.dt.year
    # FIX: hire_date masa depan (>2024, ~6.3% data sampai 2026) tidak mungkin -> NaN
    df.loc[(df["hire_year"] < 1990) | (df["hire_year"] > 2024), "hire_year"] = np.nan
    # salary_value mentah redundan dgn salary_annual (belum annualize) -> buang, pakai salary_annual/log/vs_currency
    drop = [c for c in ["employee_id","record_hash","notes","hire_date","source_dataset","import_batch","salary_raw","salary_value"] if c in df.columns]
    return df.drop(columns=drop)  # source_dataset/import_batch dibuang: bocor & tidak generalisasi


In [ ]:
df_train = df_train.drop_duplicates().reset_index(drop=True)
df = clean_df(df_train)          # train bersih (masih ada pay_band)
X_test_raw = clean_df(df_test)   # test dibersihkan dgn FUNGSI SAMA -> konsisten
print("Train bersih:", df.shape, "| Test bersih:", X_test_raw.shape)
print("\nMedian salary_value per band (bukti salary prediktif):")
print(df.groupby("pay_band")["salary_annual"].median())
print("\nMissing train:")
print(df.isnull().mean().sort_values(ascending=False).head(12).round(3))
# FIX: salary mentah tidak sebanding antar currency (GBP ~30k vs NGN ~58jt).
# Normalisasi: selisih salary_log thd median per currency (fit di TRAIN saja agar tidak bocor).
currency_median = df.groupby("currency")["salary_log"].median()
global_median = df["salary_log"].median()
df["salary_vs_currency"] = df["salary_log"] - df["currency"].map(currency_median).fillna(global_median)
X_test_raw["salary_vs_currency"] = X_test_raw["salary_log"] - X_test_raw["currency"].map(currency_median).fillna(global_median)
df.head()


## Preprocessing

`StandardScaler` untuk numerik, `OneHotEncoder(min_frequency=0.01)` untuk kategorikal agar 873 `job_title` tidak meledak jadi ribuan kolom.


In [ ]:
x = df.drop(columns=["pay_band"])
y = df["pay_band"].copy()
X_test = X_test_raw.copy()  # urutan baris sama dgn df_test (tidak di-shuffle)
print("fitur train:", x.columns.tolist())
print("fitur test :", X_test.columns.tolist())
assert x.columns.tolist() == X_test.columns.tolist(), "Fitur train/test harus sama!"
fitur_numerik = x.select_dtypes(include=np.number).columns.tolist()
fitur_kategorikal = x.select_dtypes(exclude=np.number).columns.tolist()
print("numerik:", fitur_numerik)
print("kategorikal:", fitur_kategorikal)
x_train, x_valid, y_train, y_valid = train_test_split(x, y, test_size=0.2, stratify=y, random_state=42)
numerik_pipeline = Pipeline(steps=[("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())])
kategorikal_pipeline = Pipeline(steps=[("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=0.01, sparse_output=False))])
preprocessor = ColumnTransformer(transformers=[("num", numerik_pipeline, fitur_numerik), ("cat", kategorikal_pipeline, fitur_kategorikal)])


## Modeling

KNN/SVM mentah tidak cocok untuk one-hot dimensi tinggi + 60k baris (akurasimu 0.42-0.56, `Band 4+` recall ~0). Diganti model tertimbang `class_weight='balanced'` + stacking `predict_proba` (perbaiki bug urutan kolom `rf/knn/svm` di kode lama).


In [ ]:
model_rf = Pipeline(steps=[("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1, class_weight="balanced"))])
model_rf.fit(x_train, y_train)
valid_prediction_rf = model_rf.predict(x_valid)
print("RF — akurasi:", accuracy_score(y_valid, valid_prediction_rf),
      "| f1_macro:", f1_score(y_valid, valid_prediction_rf, average="macro"))
print(classification_report(y_valid, valid_prediction_rf, zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_valid, valid_prediction_rf, cmap="Blues")
plt.show()


In [ ]:
model_hgb = Pipeline(steps=[("preprocessor", preprocessor),
    ("classifier", HistGradientBoostingClassifier(random_state=42, class_weight="balanced"))])
model_hgb.fit(x_train, y_train)
valid_prediction_hgb = model_hgb.predict(x_valid)
print("HGB — akurasi:", accuracy_score(y_valid, valid_prediction_hgb),
      "| f1_macro:", f1_score(y_valid, valid_prediction_hgb, average="macro"))
print(classification_report(y_valid, valid_prediction_hgb, zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_valid, valid_prediction_hgb, cmap="Blues")
plt.show()


In [ ]:
# Stacking BENAR: pakai predict_proba + StackingClassifier (urutan fitur dijamin, tidak manual rf/knn/svm)
estimators = [
    ("rf", Pipeline(steps=[("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1, class_weight="balanced"))])),
    ("hgb", Pipeline(steps=[("preprocessor", preprocessor),
        ("classifier", HistGradientBoostingClassifier(random_state=42, class_weight="balanced"))])),
    ("lr", Pipeline(steps=[("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced", n_jobs=-1))])),
]
stack = StackingClassifier(estimators=estimators,
    final_estimator=LogisticRegression(max_iter=1000, class_weight="balanced"),
    stack_method="predict_proba", cv=3, n_jobs=-1)
stack.fit(x_train, y_train)
final_pred = stack.predict(x_valid)
print("Stacking — akurasi:", accuracy_score(y_valid, final_pred),
      "| f1_macro:", f1_score(y_valid, final_pred, average="macro"))
print(classification_report(y_valid, final_pred, zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_valid, final_pred, cmap="Blues")
plt.show()


# Submission

`X_test` sudah hasil preprocessing dari `df_test` (jumlah & urutan baris sama). Prediksi memakai stacking di atas.


In [ ]:
ids = df_test["employee_id"].tolist()
y_test = stack.predict(X_test)
submission_df = pd.DataFrame({"employee_id": ids, "pay_band": y_test})
assert len(submission_df) == len(df_test), "Jumlah baris submission harus sama dengan test.csv!"
assert set(submission_df["pay_band"]) <= {"Band 1", "Band 2", "Band 3", "Band 4+"}, "Isi pay_band harus label asli: Band 1 / Band 2 / Band 3 / Band 4+"
submission_df.to_csv("submission.csv", index=False)
print("Distribusi prediksi:")
print(submission_df["pay_band"].value_counts())
submission_df.head()
